### Full analysis

In [1]:
import os
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau, pearsonr

# --- Configuration & Path Resolution ---
# Evaluated shot settings
SHOT_SETTINGS = [1, 5, 10, 25]

# Set evaluation source:
# 'regularized' -> reads from 'model_eval_results'
# 'epoch_tuned' -> reads from 'model_eval_results_epoch_tuned'
EVAL_SOURCE = "regularized"  

# Metric mapping: (Folder name, JSON filename, score key in JSON)
METRICS_MAP = {
    "LogME": ("logME", "all_models_logme.json", "overall_logme"),
    "H-score": ("H_score", "all_models_hscore.json", "overall_hscore"),
    "LFC": ("LFC_score", "all_models_lfc.json", "overall_lfc"),
    "LGC": ("LGC_linear", "all_models_lgc.json", "overall_lgc")
}

# Resolve workspace root depending on where the notebook is run from
# Referenced from workspace layout[cite: 2]
base_prefix = ".." if os.path.exists("../model_embeddings_eval") else "."

if EVAL_SOURCE == "regularized":
    eval_results_dir = os.path.join(base_prefix, "model_eval_results")
else:
    eval_results_dir = os.path.join(base_prefix, "model_eval_results_epoch_tuned")

output_summary_dir = os.path.join(base_prefix, "correlation_results")
os.makedirs(output_summary_dir, exist_ok=True)


def locate_metric_file(base_prefix: str, shots: int, metric_folder: str, json_filename: str):
    """
    Locates the metric JSON file, handling shot-specific directory naming conventions.
    """
    candidate_paths = [
        # Explicit shot directory: model_scores_{k}_shot/...
        os.path.join(base_prefix, f"model_scores_{shots}_shot", metric_folder, json_filename),
        # Fallback for full 25-shot stored directly in model_scores/...
        os.path.join(base_prefix, "model_scores", metric_folder, json_filename),
    ]
    for path in candidate_paths:
        if os.path.exists(path):
            return path
    return None


# --- Main Analysis Loop ---
consolidated_correlations = []
per_shot_detailed_data = {}

for shots in SHOT_SETTINGS:
    eval_file = os.path.join(eval_results_dir, f"linear_eval_{shots}_shot.json")
    
    if not os.path.exists(eval_file):
        print(f"Skipping {shots}-shot: Evaluation file not found at {eval_file}")
        continue

    # 1. Load evaluation ground-truth test accuracies
    with open(eval_file, "r") as f:
        eval_data = json.load(f)

    # Deterministic alphabetical ordering of models
    model_names = sorted(eval_data.keys())
    test_accuracies = [eval_data[m]["overall_accuracy"] for m in model_names]

    shot_df = pd.DataFrame({
        "Model": model_names,
        "Test_Accuracy": test_accuracies
    })

    # 2. Load transferability scores for all 4 metrics
    metric_scores = {}
    for metric_name, (folder, json_file, score_key) in METRICS_MAP.items():
        metric_path = locate_metric_file(base_prefix, shots, folder, json_file)
        
        if metric_path is None:
            print(f"Warning: Score file for {metric_name} ({shots}-shot) not found.")
            metric_scores[metric_name] = [np.nan] * len(model_names)
            continue

        with open(metric_path, "r") as f:
            score_data = json.load(f)

        scores = []
        for m in model_names:
            if m in score_data:
                scores.append(score_data[m].get(score_key, np.nan))
            else:
                scores.append(np.nan)
                
        metric_scores[metric_name] = scores
        shot_df[metric_name] = scores

    per_shot_detailed_data[f"{shots}_shot"] = shot_df.to_dict(orient="records")

    # 3. Print per-shot score and accuracy overview
    print(f"\n{'='*75}")
    print(f"               PERFORMANCE & METRIC VALUES ({shots}-SHOT)")
    print(f"{'='*75}")
    print(shot_df.to_string(index=False, justify="center"))

    # 4. Compute Spearman, Kendall, and Pearson Rank Correlations
    print(f"\n{'-'*75}")
    print(f"         RANK CORRELATION WITH TEST ACCURACY ({shots}-SHOT)")
    print(f"{'-'*75}")
    print(f"{'Metric':<12} | {'Spearman ρ':<12} | {'p-value':<10} | {'Kendall τ':<12} | {'p-value':<10} | {'Pearson r':<10}")
    print(f"{'-'*75}")

    for metric_name in METRICS_MAP.keys():
        vals = np.array(metric_scores[metric_name])
        accs = np.array(test_accuracies)

        valid_mask = ~np.isnan(vals) & ~np.isnan(accs)
        
        if valid_mask.sum() >= 3:
            s_rho, s_pval = spearmanr(vals[valid_mask], accs[valid_mask])
            k_tau, k_pval = kendalltau(vals[valid_mask], accs[valid_mask])
            p_r, p_pval = pearsonr(vals[valid_mask], accs[valid_mask])

            print(f"{metric_name:<12} | {s_rho:>10.4f}   | {s_pval:>8.4f} | {k_tau:>10.4f}   | {k_pval:>8.4f} | {p_r:>8.4f}")

            consolidated_correlations.append({
                "shots": shots,
                "metric": metric_name,
                "spearman_rho": float(s_rho),
                "spearman_pvalue": float(s_pval),
                "kendall_tau": float(k_tau),
                "kendall_pvalue": float(k_pval),
                "pearson_r": float(p_r),
                "pearson_pvalue": float(p_pval)
            })
        else:
            print(f"{metric_name:<12} | Insufficient data points for correlation.")

# --- Consolidated Multi-Shot Summary ---
if consolidated_correlations:
    corr_df = pd.DataFrame(consolidated_correlations)
    
    # Pivot for clean comparison across shot regimes
    pivot_spearman = corr_df.pivot(index="metric", columns="shots", values="spearman_rho")
    pivot_kendall = corr_df.pivot(index="metric", columns="shots", values="kendall_tau")

    print(f"\n{'='*75}")
    print("           CONSOLIDATED SPEARMAN RANK CORRELATION (ρ)")
    print(f"{'='*75}")
    print(pivot_spearman.round(4).to_string())

    print(f"\n{'='*75}")
    print("            CONSOLIDATED KENDALL TAU RANK CORRELATION (τ)")
    print(f"{'='*75}")
    print(pivot_kendall.round(4).to_string())

    # 5. Export results to JSON and CSV
    csv_file = os.path.join(output_summary_dir, f"rank_correlations_{EVAL_SOURCE}.csv")
    corr_df.to_csv(csv_file, index=False)

    json_file = os.path.join(output_summary_dir, f"rank_correlations_{EVAL_SOURCE}.json")
    with open(json_file, "w") as f:
        json.dump({
            "eval_source": EVAL_SOURCE,
            "correlations": consolidated_correlations,
            "model_level_data": per_shot_detailed_data
        }, f, indent=4)

    print(f"\nSaved correlation reports to:")
    print(f" - CSV:  {csv_file}")
    print(f" - JSON: {json_file}")


               PERFORMANCE & METRIC VALUES (1-SHOT)
      Model         Test_Accuracy   LogME   H-score    LFC      LGC  
          AVES-bio    0.353333    0.929706 4.999897 0.901207 0.901207
BirdAVES-biox-base    0.400000    1.121703 4.999727 0.959781 0.959781
      DistilHuBERT    0.360000    1.215707 4.999516 0.799697 0.799697
            HuBERT    0.346667    0.921762 4.999536 0.833500 0.833500
       MusicHuBERT    0.240000    1.091213 4.997757 0.924622 0.924622
             WavLM    0.286667    1.367226 4.997808 0.778391 0.778391

---------------------------------------------------------------------------
         RANK CORRELATION WITH TEST ACCURACY (1-SHOT)
---------------------------------------------------------------------------
Metric       | Spearman ρ   | p-value    | Kendall τ    | p-value    | Pearson r 
---------------------------------------------------------------------------
LogME        |     0.0857   |   0.8717 |     0.0667   |   1.0000 |  -0.2602
H-score      |  